# GNN Inference on Multi-Core CPUs and GPUs — Benchmark & Evaluation Suite

Questo notebook esegue la suite completa di benchmarking, verifica e valutazione delle prestazioni come specificato in `requirements.md` e `semantics.md`.

### Requisiti coperti:
- **`BEN-COMP-01..03`**: Confronto tra CPU Sequenziale, Multi-threaded OpenMP e GPU CUDA per **GCN** e **GraphSAGE**.
- **`BEN-COMP-04`**: Caso di scala a **1 milione di nodi** ($N=1.000.000$) e scaling di taglia del grafo.
- **`BEN-COMP-05..07`**: Studio di scalabilità lungo gli altri assi (feature width $F=32, 128$, profondità $D=2, 8$, skew gradi $S=0, 1, 2$).
- **`BEN-COMP-10..11`**: Confronto con framework esterno (**PyTorch Geometric** con `GCNConv` e `SAGEConv` standard) su CPU e CUDA.
- **`VER-01..07`**: Verifica automatica della correttezza numerica con tolleranza floating point (errore $< 10^{-4}$) e test su casi limite.
- **`BEN-MET-01..06`**: Misurazione separata di tempi di compute, memoria picco GPU/host, throughput e speedup.

## 1. Setup Ambiente, GPU e Compilazione

In [11]:
%%bash
echo '=== GPU Info ==='
nvidia-smi
echo -e '\n=== Toolchain Compilatori ==='
g++ --version | head -n 1
nvcc --version | tail -n 2


=== GPU Info ===
Mon Sep  7 03:04:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
if [ -d "$PROJECT/.git" ]; then
  git -C "$PROJECT" pull --ff-only
else
  git clone https://github.com/Alby02/cuda-gnn-inference.git "$PROJECT"
fi


fatal: Not possible to fast-forward, aborting.


CalledProcessError: Command 'b'set -euo pipefail\nPROJECT=/content/drive/MyDrive/cuda-gnn-inference\nif [ -d "$PROJECT/.git" ]; then\n  git -C "$PROJECT" pull --ff-only\nelse\n  git clone https://github.com/Alby02/cuda-gnn-inference.git "$PROJECT"\nfi\n'' returned non-zero exit status 128.

In [14]:
%pip install -q meson ninja networkit scipy ogb torch-geometric matplotlib pandas


In [15]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"

echo '=== Configurazione e Compilazione C++/CUDA (Meson + Ninja) ==='
if [ -d builddir/meson-private ]; then
  meson setup --reconfigure builddir
else
  meson setup builddir
fi
meson compile -C builddir
./builddir/gnn --help


=== Configurazione e Compilazione C++/CUDA (Meson + Ninja) ===
The Meson build system
Version: 1.12.0
Source dir: /content/drive/MyDrive/cuda-gnn-inference
Build dir: /content/drive/MyDrive/cuda-gnn-inference/builddir
Build type: native build
Project name: cuda-gnn-inference
Project version: 0.1
C++ compiler for the host machine: c++ (gcc 11.4.0 "c++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0")
C++ linker for the host machine: c++ ld.bfd 2.38
Host machine cpu family: x86_64
Host machine cpu: x86_64
Dependency openmp for host machine found: YES 4.5 (cached)
Message: OpenMP found. Enabling the parallel CLI mode.
Cuda compiler for the host machine: nvcc (nvcc 12.8.93)
Cuda linker for the host machine: nvcc nvlink 12.8.93
Message: CUDA compiler (nvcc) found. Enabling the CUDA CLI mode.
Configuring gnn_build_config.hpp using configuration
Build targets in project: 1

Found ninja-1.13.2.git.kitware.jobserver-pipe-1 at /usr/local/bin/ninja
Cleaning... 0 files.
ninja: Entering directory `/content

## 2. Smoke Test Rapido (Sanity Check)
Verifica preliminare che i tre backend (`sequential`, `parallel`, `cuda`) producano risultati equivalenti sul demo integrato.

In [16]:
import subprocess
import re
import numpy as np
from pathlib import Path

project = Path('/content/drive/MyDrive/cuda-gnn-inference')
executable = project / 'builddir' / 'gnn'

def run_mode(mode):
    res = subprocess.run([str(executable), '--backend', mode], capture_output=True, text=True, check=True)
    rows = []
    for line in res.stdout.splitlines():
        m = re.search(r'\[([^\]]+)\]', line)
        if m:
            rows.append([float(x) for x in m.group(1).split(',')])
    return np.array(rows, dtype=np.float32)

seq_out = run_mode('sequential')
par_out = run_mode('parallel')
cuda_out = run_mode('cuda')

np.testing.assert_allclose(par_out, seq_out, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(cuda_out, seq_out, atol=1e-5, rtol=1e-5)
print('OK: Smoke test completato! Sequenziale, OpenMP e CUDA coincidono perfettamente:', cuda_out.tolist())


OK: Smoke test completato! Sequenziale, OpenMP e CUDA coincidono perfettamente: [[10.0, 11.0], [25.0, 26.0]]


## 3. Validazione Correttezza su Casi Limite (`VER-03..05`)
Verifica su grafi vuoti (0 archi), nodi isolati, grafo non orientato e layer senza bias rispetto al reference matematico.

In [17]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
python synth_data/framework-validation/check_edge_cases.py ./builddir/gnn


python3: can't open file '/content/drive/MyDrive/cuda-gnn-inference/synth_data/framework-validation/check_edge_cases.py': [Errno 2] No such file or directory


CalledProcessError: Command 'b'set -euo pipefail\nPROJECT=/content/drive/MyDrive/cuda-gnn-inference\ncd "$PROJECT"\npython synth_data/framework-validation/check_edge_cases.py ./builddir/gnn\n'' returned non-zero exit status 2.

## 4. Benchmark Multilayer: Nativo (CPU/CUDA) vs. PyTorch Geometric (`BEN-COMP-10..11`)
Confronto a 3 layer per **GCN**, **GraphSAGE** e modello **Misto (GCN + GraphSAGE + GCN)**.
Valuta la correttezza numerica e misura lo speedup rispetto a PyTorch Geometric (CPU e CUDA).

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_multilayer_benchmark"
mkdir -p "$OUT_DIR"

python scripts/compare_framework.py \
  --native ./builddir/gnn \
  --graph synth_data/forward-validation/multilayer/unweighted.bin_graph \
  --features synth_data/forward-validation/multilayer/features.bin_matrix \
  --model synth_data/forward-validation/multilayer/gcn.model \
          synth_data/forward-validation/multilayer/graphsage.model \
          synth_data/forward-validation/multilayer/mixed.model \
  --backend sequential parallel cuda \
  --threads 1 2 4 \
  --block-size 128 256 \
  --warmups 2 \
  --repetitions 10 \
  --output-dir "$OUT_DIR"


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_multilayer_benchmark/comparison.csv')
if csv_path.exists():
    df = pd.read_csv(csv_path)
    cols = ['model_types', 'native_backend', 'framework_device', 'threads', 'block_size',
            'verification', 'max_abs_error', 'native_mean_ms', 'framework_mean_ms', 'native_speedup']
    display(df[cols].sort_values(by=['model_types', 'native_backend']))


## 5. Valutazione su Dataset Pubblico Reale (`DATA-PUB-01`)
Esegue il benchmark su dataset reale di riferimento (**Planetoid Cora** o **OGBN-ArXiv**).
Il comando scarica automaticamente il dataset, genera i pesi del modello ed esegue il confronto completo tra tutti i backend.

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_cora_results"

# Esecuzione su dataset Cora (per ogbn-arxiv sostituire con --dataset ogbn-arxiv)
python scripts/run_experiments.py \
  --native ./builddir/gnn \
  --dataset Cora \
  --backend sequential parallel cuda \
  # --threads omesso usa la scala automatica 2^x fino al max CPU core (es. 1 2 su Colab)
  --block-size 128 256 \
  --warmups 2 \
  --repetitions 10 \
  --output-dir "$OUT_DIR"


In [18]:
import pandas as pd
from pathlib import Path

cora_csv = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_cora_results/comparison.csv')
if cora_csv.exists():
    display(pd.read_csv(cora_csv))


## 6. Studio di Scalabilità Parametrica Multi-Asse (BEN-COMP-04..07)
Esegue lo sweep sperimentale completo variando una dimensione alla volta:
- **Nodi ($)**: 1.000, 10.000 (ordine di grandezza)
- **Feature ($)**: 32, 128 canali
- **Profondità ($)**: 2, 8 layer
- **Skew ($)**: 0 (uniforme), 1, 2 (legge di potenza)
- **Backend**: Sequential, Parallel (OpenMP: scala esponenziale ^x$ automatica fino al max di core CPU) e CUDA GPU (block size: 128 e 256)


In [19]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_scaling_results"

python scripts/run_experiments.py \
  --native ./builddir/gnn \
  --dataset none \
  --nodes 1000 10000 \
  --widths 32 128 \
  --depths 2 8 \
  --skews 0 1 2 \
  --backend sequential parallel cuda \
  # --threads omesso: auto-scaling esponenziale 2^x (1, 2 su Colab; 1, 2, 4, 8, 16 su PC)
  --block-size 128 256 \
  --warmups 2 \
  --repetitions 10 \
  --output-dir "$OUT_DIR"


python3: can't open file '/content/drive/MyDrive/cuda-gnn-inference/scripts/run_experiments.py': [Errno 2] No such file or directory


CalledProcessError: Command 'b'set -euo pipefail\nPROJECT=/content/drive/MyDrive/cuda-gnn-inference\ncd "$PROJECT"\nOUT_DIR="$PROJECT/colab_scaling_results"\n\npython scripts/run_experiments.py \\\n  --native ./builddir/gnn \\\n  --dataset none \\\n  --nodes 1000 10000 \\\n  --widths 32 128 \\\n  --depths 2 8 \\\n  --skews 0 1 2 \\\n  --backend sequential parallel cuda \\\n  # --threads omesso: auto-scaling esponenziale 2^x (1, 2 su Colab; 1, 2, 4, 8, 16 su PC)\n  --block-size 128 256 \\\n  --warmups 2 \\\n  --repetitions 10 \\\n  --output-dir "$OUT_DIR"\n'' returned non-zero exit status 2.

## 7. Large-Scale Benchmark: 1 Milione di Nodi (`BEN-COMP-04`)
Come specificato in `requirements.md` (`BEN-COMP-04`) e `project.md`:
> *"The evaluation shall measure graph-size scalability across at least one order of magnitude and shall include a million-node-scale case when permitted by the available memory..."*

Questa sezione genera un grafo sintetico a scala di **1.000.000 di nodi** (~8.000.000 di archi) ed esegue:
- Benchmark GCN e GraphSAGE su GPU CUDA (block size 128 e 256) e CPU OpenMP (2 e 4 thread)
- Confronto prestazionale con PyTorch Geometric su GPU CUDA
- Calcolo del Throughput ($	ext{nodi/s}$ ed $	ext{archi/s}$) e verifica numerica su 1 milione di nodi

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_million_nodes_results"

# Esecuzione su 1.000.000 di nodi (GPU CUDA e CPU OpenMP)
python scripts/run_experiments.py \
  --native ./builddir/gnn \
  --dataset none \
  --nodes 1000000 \
  --widths 32 \
  --depths 2 \
  --skews 0 \
  --backend parallel cuda \
  # --threads omesso: auto-scaling esponenziale 2^x
  --block-size 128 256 \
  --warmups 1 \
  --repetitions 5 \
  --output-dir "$OUT_DIR"


In [ ]:
import pandas as pd
from pathlib import Path

csv_1m = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_million_nodes_results/comparison.csv')
if csv_1m.exists():
    df_1m = pd.read_csv(csv_1m)
    print(f'=== Risultati Benchmark 1 Milione di Nodi (8M archi) ===')
    cols = ['model_types', 'native_backend', 'framework_device', 'threads', 'block_size',
            'verification', 'max_abs_error', 'native_mean_ms', 'framework_mean_ms', 'native_speedup']
    display(df_1m[cols])
else:
    print('File non trovato in:', csv_1m)


## 8. Visualizzazione Grafici e Report Finale
Mostra direttamente nel notebook i grafici di scaling generati (`plots/`) e le tabelle di sintesi.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import pandas as pd

plots_dir = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_scaling_results/plots')
if plots_dir.exists():
    pngs = sorted(plots_dir.glob('*-scaling.png')) + sorted(plots_dir.glob('*-speedup.png'))
    for p in pngs:
        print(f'\n========== {p.name} ==========')
        display(Image(filename=str(p)))
else:
    print('Nessun grafico trovato in:', plots_dir)


In [ ]:
scale_csv = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_scaling_results/comparison.csv')
if scale_csv.exists():
    df_scale = pd.read_csv(scale_csv)
    print(f'Riepilogo {len(df_scale)} configurazioni sperimentali confrontate:')
    display(df_scale[['workload', 'model_types', 'native_backend', 'framework_device', 'threads', 'block_size', 'verification', 'native_mean_ms', 'framework_mean_ms', 'native_speedup']])


# 9. GraphSAGE

In [22]:
import os
import re
import time
import struct
import subprocess
from pathlib import Path
import numpy as np

!pip install -q ogb torch torch-geometric psutil

import torch
import scipy.sparse as sp

if not hasattr(torch, '_original_torch_load'):
    torch._original_torch_load = torch.load
torch.load = lambda *args, **kwargs: torch._original_torch_load(*args, **{**kwargs, 'weights_only': False})

from ogb.nodeproppred import PygNodePropPredDataset

print("=== Downloading/Loading ogbn-arxiv dataset ===")
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='/content/dataset')
data = dataset[0]
torch.load = torch._original_torch_load

project = Path('/content/drive/MyDrive/cuda-gnn-inference')
executable = project / 'builddir' / 'gnn'
benchmark_dir = Path('/content/gnn_benchmarks')
benchmark_dir.mkdir(parents=True, exist_ok=True)

def save_bin_matrix(filepath, mat):
    mat = np.ascontiguousarray(mat, dtype=np.float32)
    with open(filepath, 'wb') as f:
        f.write(struct.pack('<QQ', np.uint64(mat.shape[0]), np.uint64(mat.shape[1])))
        f.write(mat.tobytes())

def export_csc_graph(filepath, adj):
    with open(filepath, 'wb') as f:
        num_nodes = int(adj.shape[0])
        num_edges = int(adj.nnz)
        is_directed = 1
        has_weights = 1
        f.write(struct.pack('<QQBB', num_nodes, num_edges, is_directed, has_weights))
        f.write(adj.indptr.astype(np.uint64).tobytes())
        f.write(adj.indices.astype(np.uint64).tobytes())
        f.write(adj.data.astype(np.float32).tobytes())

def export_model_manifest(filepath, layer_dims, model_type="GraphSAGE"):
    """
    Dynamically generates multi-layer model weights based on layer_dims (e.g. [128, 64, 16])
    and exports model.txt.
    """
    num_layers = len(layer_dims) - 1
    export_dir = filepath.parent
    lines = []

    for i in range(num_layers):
        d_in, d_out = layer_dims[i], layer_dims[i + 1]
        act = "RELU" if i < num_layers - 1 else "NONE"

        w_neigh_path = export_dir / f"w_neigh_l{i}_{d_in}_{d_out}.bin_matrix"
        w_self_path  = export_dir / f"w_self_l{i}_{d_in}_{d_out}.bin_matrix"
        b_path       = export_dir / f"bias_l{i}_{d_out}.bin_matrix"

        # Initialize random weights and write to binary files
        save_bin_matrix(w_neigh_path, np.random.randn(d_in, d_out).astype(np.float32) * 0.05)
        save_bin_matrix(w_self_path, np.random.randn(d_in, d_out).astype(np.float32) * 0.05)
        save_bin_matrix(b_path, np.zeros((1, d_out), dtype=np.float32))

        if model_type == "GCN":
            lines.append(f"layer GCN {act} {w_neigh_path.name} - {b_path.name}")
        else:
            lines.append(f"layer GraphSAGE {act} {w_neigh_path.name} {w_self_path.name} {b_path.name}")

    with open(filepath, 'w') as f:
        f.write(f"GNN_MODEL 1 layers {num_layers}\n")
        for line in lines:
            f.write(line + "\n")

# 3. Process monitoring and executor
def get_current_gpu_memory_mb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,nounits,noheader"],
            text=True
        )
        return float(out.strip().splitlines()[0])
    except Exception:
        return 0.0

def run_backend_bench(mode, graph_path, feat_path, model_path, warmup=1, iters=5):
    cmd = [
        str(executable), '--backend', mode,
        '--graph', str(graph_path),
        '--features', str(feat_path),
        '--model', str(model_path),
        '--repetitions', '1'
    ]

    # Warmup
    for _ in range(warmup):
        res = subprocess.run(cmd, cwd=project, capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f"{mode} Warmup failed:\n{res.stderr}")

    # Formally execute multiple iterations and take average
    latencies = []
    max_gpu_vram = 0.0
    output_rows = []

    for _ in range(iters):
        gpu_mem_before = get_current_gpu_memory_mb() if mode == 'cuda' else 0.0

        t0 = time.perf_counter()
        res = subprocess.run(cmd, cwd=project, capture_output=True, text=True)
        t1 = time.perf_counter()

        gpu_mem_after = get_current_gpu_memory_mb() if mode == 'cuda' else 0.0
        max_gpu_vram = max(max_gpu_vram, gpu_mem_after)

        if res.returncode != 0:
            raise RuntimeError(f"{mode} Execution failed:\n{res.stderr}")

        latencies.append((t1 - t0) * 1000.0)  # In milliseconds

        if not output_rows:  # Parse output once for numerical precision verification
            for line in res.stdout.splitlines():
                match = re.search(r'\[(.*?)\]', line)
                if match:
                    tokens = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', match.group(1))
                    if tokens:
                        output_rows.append([float(v) for v in tokens])

    avg_latency_ms = np.mean(latencies)
    out_tensor = np.asarray(output_rows, dtype=np.float32)
    return avg_latency_ms, out_tensor, max_gpu_vram

def run_gnn_experiments(num_nodes=5000, model_type="GraphSAGE"):
    print(f"\n=======================================================")
    print(f"      GNN Inference Performance & Memory Benchmark ({model_type})")
    print(f"=======================================================")

    edge_index = data.edge_index.numpy()
    mask = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
    sub_edges = edge_index[:, mask]
    adj = sp.csc_matrix(
        (np.ones(sub_edges.shape[1], dtype=np.float32), (sub_edges[0], sub_edges[1])),
        shape=(num_nodes, num_nodes)
    )

    graph_bin = benchmark_dir / "bench_graph.bin_graph"
    export_csc_graph(graph_bin, adj)
    print(f"Graph Topology: Nodes = {num_nodes}, Edges = {adj.nnz}\n")

    test_configs = [
        ("SmallFeat-2L", 64,  [64, 32, 16]),
        ("Nominal-2L",   128, [128, 64, 16]),
        ("LargeFeat-2L", 256, [256, 128, 16]),
        ("Nominal-3L",   128, [128, 64, 32, 16]),
        ("LargeFeat-3L", 256, [256, 128, 64, 16]),
        ("DeepNet-4L",   128, [128, 128, 64, 32, 16]),
    ]

    header = (
        f"{'Config Name':<14} | {'In Dim':<6} | {'Layers':<6} | {'Backend':<11} | "
        f"{'Latency(ms)':<11} | {'Throughput(nodes/s)':<20} | {'GPU Mem(MB)':<11} | {'Speedup':<8}"
    )
    print(header)
    print("-" * len(header))

    for name, in_dim, dims in test_configs:
        feats = np.random.randn(num_nodes, in_dim).astype(np.float32)
        feats_bin = benchmark_dir / f"feats_dim{in_dim}.bin_matrix"
        save_bin_matrix(feats_bin, feats)

        model_desc = benchmark_dir / f"model_{name}.txt"
        export_model_manifest(model_desc, dims, model_type=model_type)

        results = {}
        for backend in ['sequential', 'parallel', 'cuda']:
            try:
                lat_ms, out_mat, vram = run_backend_bench(
                    backend, graph_bin, feats_bin, model_desc, warmup=1, iters=3
                )
                throughput = (num_nodes / (lat_ms / 1000.0))
                results[backend] = (lat_ms, throughput, vram, out_mat)
            except Exception as e:
                err_msg = str(e).strip().splitlines()[-1]
                print(f"{name:<14} | {in_dim:<6} | {len(dims)-1:<6} | {backend:<11} | Error: {err_msg[:30]}")

        seq_lat = results.get('sequential', (None,))[0]

        for backend in ['sequential', 'parallel', 'cuda']:
            if backend not in results:
                continue
            lat_ms, throughput, vram, _ = results[backend]
            vram_str = f"{vram:.1f}" if backend == 'cuda' else "-"

            if seq_lat and lat_ms > 0:
                speedup_str = f"{seq_lat / lat_ms:.2f}x" if backend != 'sequential' else "1.00x"
            else:
                speedup_str = "-"

            print(
                f"{name:<14} | {in_dim:<6} | {len(dims)-1:<6} | {backend:<11} | "
                f"{lat_ms:<11.2f} | {throughput:<20.1f} | {vram_str:<11} | {speedup_str:<8}"
            )

        if 'sequential' in results and 'parallel' in results:
            np.testing.assert_allclose(results['parallel'][3], results['sequential'][3], rtol=1e-4, atol=1e-4)
        if 'sequential' in results and 'cuda' in results:
            np.testing.assert_allclose(results['cuda'][3], results['sequential'][3], rtol=1e-4, atol=1e-4)
        print("-" * len(header))

run_gnn_experiments(num_nodes=3000, model_type="GraphSAGE")

=== Downloading/Loading ogbn-arxiv dataset ===

      GNN Inference Performance & Memory Benchmark (GraphSAGE)
Graph Topology: Nodes = 3000, Edges = 468

Config Name    | In Dim | Layers | Backend     | Latency(ms) | Throughput(nodes/s)  | GPU Mem(MB) | Speedup 
------------------------------------------------------------------------------------------------------------
SmallFeat-2L   | 64     | 2      | sequential  | 26.61       | 112741.5             | -           | 1.00x   
SmallFeat-2L   | 64     | 2      | parallel    | 31.85       | 94191.9              | -           | 0.84x   
SmallFeat-2L   | 64     | 2      | cuda        | 255.61      | 11736.7              | 3.0         | 0.10x   
------------------------------------------------------------------------------------------------------------
Nominal-2L     | 128    | 2      | sequential  | 69.45       | 43196.0              | -           | 1.00x   
Nominal-2L     | 128    | 2      | parallel    | 69.42       | 43212.7             

In [23]:
import os
import re
import time
import struct
import subprocess
import threading
from pathlib import Path
import numpy as np

!pip install -q ogb torch torch-geometric

import torch
import scipy.sparse as sp

if not hasattr(torch, '_original_torch_load'):
    torch._original_torch_load = torch.load
torch.load = lambda *args, **kwargs: torch._original_torch_load(*args, **{**kwargs, 'weights_only': False})

from ogb.nodeproppred import PygNodePropPredDataset

print("=== Downloading/Loading ogbn-arxiv dataset ===")
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='/content/dataset')
data = dataset[0]
torch.load = torch._original_torch_load

project = Path('/content/drive/MyDrive/cuda-gnn-inference')
executable = project / 'builddir' / 'gnn'
benchmark_dir = Path('/content/gnn_stress_benchmarks')
benchmark_dir.mkdir(parents=True, exist_ok=True)

# 2. Binary export utility functions
def save_bin_matrix(filepath, mat):
    mat = np.ascontiguousarray(mat, dtype=np.float32)
    with open(filepath, 'wb') as f:
        f.write(struct.pack('<QQ', np.uint64(mat.shape[0]), np.uint64(mat.shape[1])))
        f.write(mat.tobytes())

def export_csc_graph(filepath, adj):
    with open(filepath, 'wb') as f:
        num_nodes = int(adj.shape[0])
        num_edges = int(adj.nnz)
        is_directed = 1
        has_weights = 1
        f.write(struct.pack('<QQBB', num_nodes, num_edges, is_directed, has_weights))
        f.write(adj.indptr.astype(np.uint64).tobytes())
        f.write(adj.indices.astype(np.uint64).tobytes())
        f.write(adj.data.astype(np.float32).tobytes())

def export_model_manifest(filepath, layer_dims, model_type="GraphSAGE"):
    num_layers = len(layer_dims) - 1
    export_dir = filepath.parent
    lines = []

    for i in range(num_layers):
        d_in, d_out = layer_dims[i], layer_dims[i + 1]
        act = "RELU" if i < num_layers - 1 else "NONE"

        w_neigh = export_dir / f"w_neigh_l{i}_{d_in}_{d_out}.bin_matrix"
        w_self  = export_dir / f"w_self_l{i}_{d_in}_{d_out}.bin_matrix"
        b_path  = export_dir / f"bias_l{i}_{d_out}.bin_matrix"

        save_bin_matrix(w_neigh, np.random.randn(d_in, d_out).astype(np.float32) * 0.05)
        save_bin_matrix(w_self, np.random.randn(d_in, d_out).astype(np.float32) * 0.05)
        save_bin_matrix(b_path, np.zeros((1, d_out), dtype=np.float32))

        if model_type == "GCN":
            lines.append(f"layer GCN {act} {w_neigh.name} - {b_path.name}")
        else:
            lines.append(f"layer GraphSAGE {act} {w_neigh.name} {w_self.name} {b_path.name}")

    with open(filepath, 'w') as f:
        f.write(f"GNN_MODEL 1 layers {num_layers}\n")
        for line in lines:
            f.write(line + "\n")

class VramMonitor:
    def __init__(self, interval_sec=0.01):
        self.interval = interval_sec
        self.max_vram = 0.0
        self._running = False
        self._thread = None

    def _poll(self):
        while self._running:
            try:
                mem = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,nounits,noheader"],
                    text=True
                )
                val = float(mem.strip().splitlines()[0])
                if val > self.max_vram:
                    self.max_vram = val
            except Exception:
                pass
            time.sleep(self.interval)

    def start(self):
        self.max_vram = 0.0
        self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=0.2)
        return self.max_vram

def run_backend_e2e(mode, graph_path, feat_path, model_path, repetitions=5, threads=2):
    csv_output_path = benchmark_dir / f"result_{mode}.csv"
    if csv_output_path.exists():
        csv_output_path.unlink()

    cmd = [
        str(executable), '--backend', mode,
        '--graph', str(graph_path),
        '--features', str(feat_path),
        '--model', str(model_path),
        '--repetitions', str(repetitions),
        '--threads', str(threads),
        '--output', str(csv_output_path)
    ]

    monitor = VramMonitor()
    if mode == 'cuda':
        monitor.start()

    t0 = time.perf_counter()
    res = subprocess.run(cmd, cwd=project, capture_output=True, text=True)
    t1 = time.perf_counter()

    peak_vram = monitor.stop() if mode == 'cuda' else 0.0

    if res.returncode != 0:
        raise RuntimeError(f"{mode} crashed (exit code {res.returncode}):\n{res.stderr}")

    total_e2e_ms = (t1 - t0) * 1000.0
    per_run_e2e_ms = total_e2e_ms / repetitions

    native_compute_ms = None
    native_stddev_ms = 0.0
    if csv_output_path.exists():
        try:
            with open(csv_output_path, 'r') as f:
                lines = [line.strip() for line in f if line.strip()]
                if len(lines) >= 2:
                    header = [h.strip() for h in lines[0].split(',')]
                    first_row = [v.strip() for v in lines[1].split(',')]
                    data_map = dict(zip(header, first_row))

                    if 'mean_ms' in data_map and data_map['mean_ms']:
                        native_compute_ms = float(data_map['mean_ms'])
                    if 'stddev_ms' in data_map and data_map['stddev_ms']:
                        native_stddev_ms = float(data_map['stddev_ms'])
        except Exception:
            pass

    if native_compute_ms is None:
        match = re.search(
            r"compute\s+mean\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*ms"
            r"(?:;\s*population\s+stddev\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*ms)?",
            res.stdout,
            re.IGNORECASE
        )
        if match:
            native_compute_ms = float(match.group(1))
            if match.group(2):
                native_stddev_ms = float(match.group(2))

    output_rows = []
    for line in res.stdout.splitlines():
        m = re.search(r'\[(.*?)\]', line)
        if m:
            tokens = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', m.group(1))
            if tokens:
                output_rows.append([float(v) for v in tokens])

    return per_run_e2e_ms, native_compute_ms, native_stddev_ms, np.asarray(output_rows, dtype=np.float32), peak_vram

def run_heavy_gnn_benchmark(num_nodes=80000, model_type="GraphSAGE", repetitions=5, threads=2):
    print(f"\n=======================================================================================================================")
    print(f"      GNN Comprehensive Benchmark (Native C++ Compute vs. End-to-End Latency, {model_type}, Reps={repetitions})")
    print(f"=======================================================================================================================")

    # Extract large subgraph (80,000 nodes, ~1M edges)
    edge_index = data.edge_index.numpy()
    mask = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
    sub_edges = edge_index[:, mask]
    adj = sp.csc_matrix(
        (np.ones(sub_edges.shape[1], dtype=np.float32), (sub_edges[0], sub_edges[1])),
        shape=(num_nodes, num_nodes)
    )

    graph_bin = benchmark_dir / f"heavy_graph_{num_nodes}.bin_graph"
    export_csc_graph(graph_bin, adj)
    print(f"Stress Graph Topology: Nodes = {num_nodes:,} | Edges = {adj.nnz:,}\n")

    # Scaling configurations across depth and feature dimension
    test_configs = [
        ("Standard-2L",   128, [128, 64, 16]),
        ("LargeFeat-2L",  256, [256, 128, 16]),
        ("WideHidden-2L", 128, [128, 256, 64]),
        ("DeepNet-3L",    128, [128, 128, 64, 16]),
        ("LargeDeep-3L",  256, [256, 128, 64, 16]),
    ]

    header = (
        f"{'Config Name':<14} | {'Dim':<5} | {'L':<2} | {'Backend':<11} | "
        f"{'Native C++(ms)':<17} | {'E2E Lat(ms)':<12} | {'Throughput(nodes/s)':<20} | "
        f"{'Peak VRAM(MB)':<13} | {'C++ Speedup'}"
    )
    print(header)
    print("-" * len(header))

    for name, in_dim, dims in test_configs:
        feats = np.random.randn(num_nodes, in_dim).astype(np.float32)
        feats_bin = benchmark_dir / f"feats_{num_nodes}_dim{in_dim}.bin_matrix"
        save_bin_matrix(feats_bin, feats)

        model_desc = benchmark_dir / f"model_{name}.txt"
        export_model_manifest(model_desc, dims, model_type=model_type)

        results = {}
        for backend in ['sequential', 'parallel', 'cuda']:
            try:
                e2e_ms, native_ms, native_std, out_mat, vram = run_backend_e2e(
                    backend, graph_bin, feats_bin, model_desc, repetitions=repetitions, threads=threads
                )
                effective_ms = native_ms if (native_ms is not None and native_ms > 0) else e2e_ms
                throughput = num_nodes / (effective_ms / 1000.0)
                results[backend] = (e2e_ms, native_ms, native_std, throughput, vram, out_mat)
            except Exception as e:
                err_msg = str(e).strip().splitlines()[-1]
                print(f"{name:<14} | {in_dim:<5} | {len(dims)-1:<2} | {backend:<11} | Error: {err_msg[:35]}")

        seq_native = results.get('sequential', (None, None))[1]
        seq_e2e = results.get('sequential', (None, None))[0]
        seq_baseline = seq_native if (seq_native is not None and seq_native > 0) else seq_e2e

        for backend in ['sequential', 'parallel', 'cuda']:
            if backend not in results:
                continue
            e2e_ms, native_ms, native_std, throughput, vram, _ = results[backend]
            vram_str = f"{vram:.1f}" if backend == 'cuda' else "-"

            if native_ms is not None:
                native_str = f"{native_ms:.2f} ± {native_std:.2f}" if native_std > 0 else f"{native_ms:.2f}"
                current_time = native_ms
            else:
                native_str = f"~{e2e_ms:.2f} (E2E)"
                current_time = e2e_ms

            if seq_baseline and current_time and backend != 'sequential':
                speedup = f"{seq_baseline / current_time:.2f}x"
            else:
                speedup = "1.00x"

            print(
                f"{name:<14} | {in_dim:<5} | {len(dims)-1:<2} | {backend:<11} | "
                f"{native_str:<17} | {e2e_ms:<12.2f} | {throughput:<20.1f} | "
                f"{vram_str:<13} | {speedup}"
            )

        # Cross-backend precision consistency check
        if 'sequential' in results and 'parallel' in results:
            np.testing.assert_allclose(results['parallel'][5], results['sequential'][5], rtol=1e-4, atol=1e-4)
        if 'sequential' in results and 'cuda' in results:
            np.testing.assert_allclose(results['cuda'][5], results['sequential'][5], rtol=1e-4, atol=1e-4)
        print("-" * len(header))

# Execute benchmark with 80,000 nodes and 2 OpenMP threads
run_heavy_gnn_benchmark(num_nodes=80000, model_type="GraphSAGE", repetitions=5, threads=2)

=== Downloading/Loading ogbn-arxiv dataset ===

      GNN Comprehensive Benchmark (Native C++ Compute vs. End-to-End Latency, GraphSAGE, Reps=5)
Stress Graph Topology: Nodes = 80,000 | Edges = 270,449

Config Name    | Dim   | L  | Backend     | Native C++(ms)    | E2E Lat(ms)  | Throughput(nodes/s)  | Peak VRAM(MB) | C++ Speedup
---------------------------------------------------------------------------------------------------------------------------------
Standard-2L    | 128   | 2  | sequential  | 883.88 ± 62.98    | 1132.86      | 90509.9              | -             | 1.00x
Standard-2L    | 128   | 2  | parallel    | 539.91 ± 11.68    | 684.39       | 148173.1             | -             | 1.64x
Standard-2L    | 128   | 2  | cuda        | ~80.60 (E2E)      | 80.60        | 992559.0             | 311.0         | 10.97x
---------------------------------------------------------------------------------------------------------------------------------
LargeFeat-2L   | 256   | 2  | seque